<a href="https://colab.research.google.com/github/munnurumahesh03-coder/Enterprise-RAG-Document-AI/blob/main/Enterprise_RAG_Architecture_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Cell 1: Install RAG Dependencies
!pip install -q langchain langchain-community langchain-groq chromadb sentence-transformers pypdf streamlit
print("✅ Enterprise RAG Libraries Installed!")

✅ Enterprise RAG Libraries Installed!


In [3]:
# Cell 2: Download the AI Research Paper
!wget -q -O attention_paper.pdf https://arxiv.org/pdf/1706.03762.pdf
print("✅ PDF Downloaded Successfully: attention_paper.pdf" )

✅ PDF Downloaded Successfully: attention_paper.pdf


In [4]:
# Cell 3: Load and Chunk the Document (FIXED)
!pip install -q langchain-text-splitters

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("⏳ Reading the PDF and chunking the text...")

# 1. Load the PDF
loader = PyPDFLoader("attention_paper.pdf")
pages = loader.load()
print(f"📄 Successfully loaded {len(pages)} pages.")

# 2. Initialize the Text Splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

# 3. Chop the pages into chunks
chunks = text_splitter.split_documents(pages)

print(f"✂️ Document successfully split into {len(chunks)} chunks!")
print(f"🔍 Preview of Chunk 1:\n{chunks[0].page_content[:200]}...")

/tmp/ipykernel_2535/1211302659.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


⏳ Reading the PDF and chunking the text...
📄 Successfully loaded 15 pages.
✂️ Document successfully split into 52 chunks!
🔍 Preview of Chunk 1:
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
...


In [5]:
# Cell 4: Embeddings and Vector Database (ChromaDB)
!pip install -q langchain-huggingface

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

print("🧠 Downloading Embedding Model (Turning text into math)...")
# We use a fast, open-source embedding model from HuggingFace
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

print("🗄️ Building the Chroma Vector Database...")
# This takes our 52 chunks, embeds them, and stores them in a local ChromaDB
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

print("✅ Vector Database built successfully! The math is ready to be searched.")

🧠 Downloading Embedding Model (Turning text into math)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

🗄️ Building the Chroma Vector Database...
✅ Vector Database built successfully! The math is ready to be searched.


In [7]:
# Cell 5: The RAG Pipeline (Connecting ChromaDB to Groq)
import os
import getpass
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Secure API Key
if "GROQ_API_KEY" not in os.environ:
    print("🔑 Enter your Groq API Key:")
    os.environ["GROQ_API_KEY"] = getpass.getpass()

print("🧠 Waking up the LLM...")
# Using the Qwen model we know works flawlessly
llm = ChatGroq(temperature=0, model_name="qwen/qwen3.8-27b")

# 2. Turn ChromaDB into a Retriever (Fetch top 3 chunks)
retriever = vector_db.as_retriever(search_kwargs={"k": 3})

# 3. The Strict RAG Prompt
template = """
You are an elite AI Research Assistant. Answer the question based ONLY on the following context.
If the answer is not in the context, say "I cannot answer this based on the provided document."
Do not hallucinate.

Context:
{context}

Question: {question}

Answer:
"""
prompt = ChatPromptTemplate.from_template(template)

# 4. Build the LCEL (LangChain Expression Language) Chain
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
print("✅ Enterprise RAG Pipeline is LIVE!")

# 5. Ask a highly technical question about the paper!
question = "Why is self-attention better than recurrent neural networks (RNNs) according to the paper?"
print(f"\n🗣️ USER: {question}\n")

answer = rag_chain.invoke(question)
print(f"🎯 FINAL ANSWER:\n{answer}")

🧠 Waking up the LLM...
✅ Enterprise RAG Pipeline is LIVE!

🗣️ USER: Why is self-attention better than recurrent neural networks (RNNs) according to the paper?

🎯 FINAL ANSWER:
According to the provided context, self-attention is better than recurrent neural networks (RNNs) in terms of **computational complexity** and **parallelization**:

1.  **Computational Complexity:** Self-attention layers are faster than recurrent layers when the sequence length is short (implied by the comparison of O(1) vs O(n) operations per step, though the specific O(1) is cut off, the text states self-attention is faster in terms of computational complexity).
2.  **Parallelization:** Self-attention allows for more computation to be parallelized. A recurrent layer requires **O(n) sequential operations**, whereas self-attention reduces the number of sequential operations (the text notes that in the Transformer, the distance between positions is reduced to a **constant number of operations**).

The paper motiva

In [17]:
%%writefile app.py
import streamlit as st
import os
import tempfile
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# --- UI Configuration ---
st.set_page_config(page_title="Enterprise Document AI", page_icon="📄", layout="wide")
st.title("📄 Enterprise Document AI & RAG Engine")
st.markdown("Upload any PDF document and ask questions. The AI will extract the exact context and provide hallucination-free answers.")

# Fetch API key securely from Streamlit Secrets
# (This ensures the user never has to type it!)
GROQ_API_KEY = st.secrets["GROQ_API_KEY"]

# --- Sidebar: Setup & Upload ---
with st.sidebar:
    st.header("📂 Document Upload")
    uploaded_file = st.file_uploader("Upload a PDF", type=["pdf"])

    if st.button("Process Document"):
        if not uploaded_file:
            st.error("⚠️ Please upload a PDF document.")
        else:
            with st.spinner("Chunking and Embedding Document..."):
                with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp_file:
                    tmp_file.write(uploaded_file.getvalue())
                    tmp_path = tmp_file.name

                loader = PyPDFLoader(tmp_path)
                pages = loader.load()
                text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
                chunks = text_splitter.split_documents(pages)

                embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
                vector_db = Chroma.from_documents(documents=chunks, embedding=embeddings)

                st.session_state.vector_db = vector_db
                st.success("✅ Document processed successfully! You can now chat.")

# --- Helper Function ---
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# --- Main Chat Interface ---
if "messages" not in st.session_state:
    st.session_state.messages = []

for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

if prompt := st.chat_input("Ask a question about your document..."):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    with st.chat_message("assistant"):
        if "vector_db" not in st.session_state:
            st.warning("⚠️ Please upload and process a PDF document first.")
        else:
            with st.spinner("Searching document..."):
                llm = ChatGroq(temperature=0, model_name="qwen/qwen3.8-27b", api_key=GROQ_API_KEY, max_tokens=500)
                retriever = st.session_state.vector_db.as_retriever(search_kwargs={"k": 3})

                template = """
                You are an elite AI Research Assistant. Answer the question based ONLY on the following context.
                If the answer is not in the context, say "I cannot answer this based on the provided document."
                Do not hallucinate.

                Context:
                {context}

                Question: {question}

                Answer:
                """
                prompt_template = ChatPromptTemplate.from_template(template)

                rag_chain = (
                    {"context": retriever | format_docs, "question": RunnablePassthrough()}
                    | prompt_template
                    | llm
                    | StrOutputParser()
                )

                response = rag_chain.invoke(prompt)
                st.markdown(response)
                st.session_state.messages.append({"role": "assistant", "content": response})

Overwriting app.py


In [20]:
import importlib.metadata
from google.colab import files

# 1. Define only the packages our RAG app actually uses
core_packages = [
    "streamlit",
    "langchain",
    "langchain-community",
    "langchain-groq",
    "langchain-huggingface",
    "langchain-text-splitters",
    "chromadb",
    "sentence-transformers",
    "pypdf"
]

# 2. Dynamically fetch their exact installed versions
requirements_lines = []
for pkg in core_packages:
    try:
        version = importlib.metadata.version(pkg)
        requirements_lines.append(f"{pkg}=={version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"⚠️ Warning: {pkg} is not installed!")

# 3. Add the Streamlit Cloud specific fix for ChromaDB
requirements_lines.append("pysqlite3-binary")

# 4. Write to requirements.txt
with open("requirements.txt", "w") as f:
    f.write("\n".join(requirements_lines))

print("✅ requirements.txt generated dynamically:\n")
print("\n".join(requirements_lines))

# 5. Download the required files to your laptop
print("\n📥 Downloading files to your computer...")
files.download("app.py")
files.download("requirements.txt")


✅ requirements.txt generated dynamically:

streamlit==1.63.0
langchain==1.3.17
langchain-community==0.4.2
langchain-groq==1.1.3
langchain-huggingface==1.2.2
langchain-text-splitters==1.1.2
chromadb==1.5.9
sentence-transformers==5.7.0
pypdf==6.17.0
pysqlite3-binary

📥 Downloading files to your computer...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>